In [0]:
dbutils.widgets.text("catalog",          "retail_catalog", "Catalog")
dbutils.widgets.text("source_schema",    "bronze",         "Source Schema")
dbutils.widgets.text("target_schema",    "silver",         "Target Schema")
dbutils.widgets.text("source_table",     "raw_products",   "Source Table")
dbutils.widgets.dropdown("load_mode",    "incremental", ["full", "incremental"], "Load Mode")

In [0]:
CATALOG       = dbutils.widgets.get("catalog")
SRC_SCHEMA    = dbutils.widgets.get("source_schema")
TGT_SCHEMA    = dbutils.widgets.get("target_schema")
SRC_TABLE     = dbutils.widgets.get("source_table")
LOAD_MODE     = dbutils.widgets.get("load_mode")
 
BRONZE_TABLE  = f"{CATALOG}.{SRC_SCHEMA}.{SRC_TABLE}"
SILVER_PRODUCTS   = f"{CATALOG}.{TGT_SCHEMA}.products"
SILVER_SNAPSHOT   = f"{CATALOG}.{TGT_SCHEMA}.inventory_snapshot"
SILVER_REVIEWS    = f"{CATALOG}.{TGT_SCHEMA}.product_reviews"
WATERMARK_TABLE   = f"{CATALOG}.{SRC_SCHEMA}.watermark_log"
 
print(f"Source : {BRONZE_TABLE}")
print(f"Targets: {SILVER_PRODUCTS}, {SILVER_SNAPSHOT}, {SILVER_REVIEWS}")
print(f"Mode   : {LOAD_MODE}")

In [0]:
from datetime import datetime, timezone
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, lit, trim, upper, lower, when, coalesce,
    round as spark_round, current_timestamp, to_timestamp,
    explode, regexp_replace, row_number, avg, count
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
def clean_string(df: DataFrame, column: str, default: str = "UNKNOWN") -> DataFrame:
    """Trims whitespace and replaces nulls/empty strings with default."""
    return df.withColumn(
        column,
        when(col(column).isNull() | (trim(col(column)) == ""), lit(default))
        .otherwise(trim(col(column)))
    )
 
def standardise_category(df: DataFrame) -> DataFrame:
    """Converts category to UPPER_SNAKE_CASE. e.g. 'beauty products' → 'BEAUTY_PRODUCTS'"""
    return df.withColumn(
        "category",
        upper(regexp_replace(trim(col("category")), r"\s+", "_"))
    )
 
def standardise_status(df: DataFrame) -> DataFrame:
    """Maps raw availability strings to controlled vocabulary."""
    return df.withColumn(
        "availability_status",
        when(lower(col("availability_status")) == "in stock",     "IN_STOCK")
        .when(lower(col("availability_status")) == "out of stock", "OUT_OF_STOCK")
        .when(lower(col("availability_status")) == "low stock",    "LOW_STOCK")
        .otherwise("UNKNOWN")
    )

In [0]:
def add_stock_flags(df: DataFrame,
                    low_threshold: int = 10,
                    overstock_threshold: int = 100) -> DataFrame:
    return (
        df
        .withColumn("stock_level",
            when(col("stock") == 0,                   "OUT_OF_STOCK")
            .when(col("stock") <= 5,                  "CRITICAL")
            .when(col("stock") <= low_threshold,      "LOW")
            .when(col("stock") > overstock_threshold, "OVERSTOCK")
            .otherwise("NORMAL")
        )
        .withColumn("is_low_stock",  col("stock") <= lit(low_threshold))
        .withColumn("is_overstock",  col("stock") >  lit(overstock_threshold))
    )

In [0]:
def add_computed_price(df: DataFrame) -> DataFrame:
    return df.withColumn(
        "discounted_price",
        spark_round(col("price") * (1 - col("discount_pct") / 100), 2)
    )

In [0]:
def add_stock_value(df: DataFrame) -> DataFrame:
    """Computes stock_value = stock * discounted_price."""
    return df.withColumn(
        "stock_value",
        spark_round(col("stock") * col("discounted_price"), 2)
    )

In [0]:
def upsert_delta(target_table: str, source_df: DataFrame, merge_key: str) -> None:
    target = DeltaTable.forName(spark, target_table)
    (
        target.alias("t")
        .merge(source_df.alias("s"), f"t.{merge_key} = s.{merge_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"   Upserted → {target_table}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType

def get_watermark(catalog: str, schema: str, table_name: str) -> str:
    """Reads last watermark for incremental filtering. Returns epoch start if first run."""
    try:
        result = spark.sql(f"""
            SELECT MAX(last_updated_at) AS wm
            FROM {catalog}.{schema}.watermark_log
            WHERE table_name = '{table_name}'
        """).collect()[0]["wm"]
        return result.isoformat() if result else "1900-01-01T00:00:00Z"
    except Exception:
        return "1900-01-01T00:00:00Z"
 
def save_watermark(catalog: str, schema: str, table_name: str, new_wm, count: int) -> None:
    """Saves the latest _updated_at as the new watermark after a successful Silver run."""
    wm_schema = StructType([
        StructField("table_name",      StringType(),    True),
        StructField("last_updated_at", TimestampType(), True),
        StructField("load_mode",       StringType(),    True),
        StructField("records_loaded",  IntegerType(),   True),
        StructField("run_at",          TimestampType(), True)
    ])
    wm_data = [(
        table_name,
        new_wm,
        LOAD_MODE,
        count,
        datetime.now(timezone.utc)
    )]
    wm_df = spark.createDataFrame(wm_data, schema=wm_schema)
    wm_df.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.watermark_log")
    print(f"  Watermark saved → {new_wm}")
 
print(" Functions registered")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TGT_SCHEMA}")
 
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_PRODUCTS} (
        product_id             INT,
        title                  STRING,
        description            STRING,
        category               STRING,
        brand                  STRING,
        sku                    STRING,
        price                  DOUBLE,
        discount_pct           DOUBLE,
        discounted_price       DOUBLE,
        stock                  INT,
        stock_level            STRING,
        is_low_stock           BOOLEAN,
        is_overstock           BOOLEAN,
        stock_value            DOUBLE,
        weight                 DOUBLE,
        width                  DOUBLE,
        height                 DOUBLE,
        depth                  DOUBLE,
        availability_status    STRING,
        warranty_information   STRING,
        shipping_information   STRING,
        return_policy          STRING,
        minimum_order_qty      INT,
        barcode                STRING,
        thumbnail              STRING,
        created_at             TIMESTAMP,
        updated_at             TIMESTAMP,
        _updated_at            TIMESTAMP,
        _silver_processed_at   TIMESTAMP,
        _layer                 STRING
    )
    USING DELTA
    PARTITIONED BY (category)
    COMMENT 'Silver: Cleansed product master with stock flags'
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f" {SILVER_PRODUCTS}")
 
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SNAPSHOT} (
        product_id           INT,
        sku                  STRING,
        title                STRING,
        category             STRING,
        stock                INT,
        stock_level          STRING,
        is_low_stock         BOOLEAN,
        is_overstock         BOOLEAN,
        stock_value          DOUBLE,
        availability_status  STRING,
        snapshot_ts          TIMESTAMP,
        _layer               STRING
    )
    USING DELTA
    PARTITIONED BY (category)
    COMMENT 'Silver: Append-only inventory snapshots per run'
""")
print(f" {SILVER_SNAPSHOT}")
 
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_REVIEWS} (
        product_id      INT,
        sku             STRING,
        rating          DOUBLE,
        comment         STRING,
        review_date     TIMESTAMP,
        reviewer_name   STRING,
        reviewer_email  STRING,
        _silver_processed_at TIMESTAMP,
        _layer          STRING
    )
    USING DELTA
    COMMENT 'Silver: Exploded product reviews'
""")
print(f" {SILVER_REVIEWS}")

In [0]:
def run_silver_load():
    """
    Reads from bronze, applies transformations, and writes to all three Silver tables.
    Incremental mode filters bronze records newer than last silver watermark.
    """
    # ── Read Bronze ───────────────────────────────────────────
    bronze_df = spark.table(BRONZE_TABLE)
 
    if LOAD_MODE == "incremental":
        last_wm = get_watermark(CATALOG, SRC_SCHEMA, SILVER_PRODUCTS)
        print(f"  Incremental from: {last_wm}")
        bronze_df = bronze_df.filter(col("_updated_at") > lit(last_wm).cast("timestamp"))
 
    # Deduplicate: latest record per product_id
    w = Window.partitionBy("id").orderBy(col("_ingested_at").desc())
    bronze_df = (
        bronze_df
        .withColumn("_rn", row_number().over(w))
        .filter(col("_rn") == 1).drop("_rn")
    )
 
    count = bronze_df.count()
    if count == 0:
        print("  No new records — skipping Silver write")
        return
    print(f"  Processing {count} records")
 
    # ── Flatten & Transform ───────────────────────────────────
    df = (
        bronze_df.select(
            col("id").alias("product_id"),
            "title", "description", "category", "brand", "sku",
            "price",
            col("discountPercentage").alias("discount_pct"),
            "stock", "weight",
            col("dimensions.width").alias("width"),
            col("dimensions.height").alias("height"),
            col("dimensions.depth").alias("depth"),
            col("availabilityStatus").alias("availability_status"),
            col("warrantyInformation").alias("warranty_information"),
            col("shippingInformation").alias("shipping_information"),
            col("returnPolicy").alias("return_policy"),
            col("minimumOrderQuantity").alias("minimum_order_qty"),
            col("meta.barcode").alias("barcode"),
            to_timestamp(col("meta.createdAt")).alias("created_at"),
            to_timestamp(col("meta.updatedAt")).alias("updated_at"),
            "thumbnail", "_updated_at",
        )
    )
 
    for c in ["title", "brand", "sku", "warranty_information", "shipping_information"]:
        df = clean_string(df, c)
 
    df = standardise_category(df)
    df = standardise_status(df)
    df = add_stock_flags(df)
    df = add_computed_price(df)
    df = add_stock_value(df)
    df = (
        df
        .withColumn("_silver_processed_at", current_timestamp())
        .withColumn("_layer", lit("silver"))
    )
 
    # ── Write: Products (MERGE upsert) ────────────────────────
    upsert_delta(SILVER_PRODUCTS, df, "product_id")
 
    # ── Write: Snapshot (APPEND) ──────────────────────────────
    snapshot_df = (
        df.select(
            "product_id", "sku", "title", "category",
            "stock", "stock_level", "is_low_stock", "is_overstock",
            "stock_value", "availability_status",
            current_timestamp().alias("snapshot_ts"),
            lit("silver").alias("_layer"),
        )
    )
    snapshot_df.write.format("delta").mode("append").saveAsTable(SILVER_SNAPSHOT)
    print(f"   Snapshot appended → {SILVER_SNAPSHOT}")
 
    # ── Write: Reviews (OVERWRITE — fixed set) ────────────────
    reviews_df = (
        bronze_df
        .select("id", "sku", explode(col("reviews")).alias("r"))
        .select(
            col("id").alias("product_id"), col("sku"),
            col("r.rating").alias("rating"),
            col("r.comment").alias("comment"),
            to_timestamp(col("r.date")).alias("review_date"),
            col("r.reviewerName").alias("reviewer_name"),
            col("r.reviewerEmail").alias("reviewer_email"),
            current_timestamp().alias("_silver_processed_at"),
            lit("silver").alias("_layer"),
        )
    )
    reviews_df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(SILVER_REVIEWS)
    print(f"   Reviews written → {SILVER_REVIEWS}")
 
    # ── Save Watermark
    new_wm = df.selectExpr("MAX(_updated_at)").collect()[0][0]
    save_watermark(CATALOG, SRC_SCHEMA, SILVER_PRODUCTS, new_wm, count)
 
    print("\n Silver load complete!")

In [0]:
run_silver_load()

In [0]:
%sql
Select * from retail_catalog.silver.products

In [0]:
%sql
Select * from retail_catalog.silver.inventory_snapshot

In [0]:
%sql
Select * from  retail_catalog.silver.product_reviews